In [3]:
# ================================================================
# TORCH: Factory-Level Risk Index — Myanmar Apparel Supply Chains
# Stage 5: Risk Scoring & Benchmarking
# ================================================================
# Goal: Aggregate article-level predictions to the factory level,
# apply ILO-based weights following the Better Work CAT cluster
# structure, and produce a final composite risk index per factory.
#
# Source: Better Work Global Compliance Assessment Tool (ILO & IFC, 2025)
# https://betterwork.org/wp-content/uploads/Better-Work-Global-Compliance-Assessment-Tool.pdf

import pandas as pd
import numpy as np

news  = pd.read_csv("stage4_news_predictions.csv")
osh   = pd.read_csv("stage1_osh.csv")
bhrrc = pd.read_csv("stage1_bhrrc_labels.csv")

pred_cols = [
    "pred_child_labour", "pred_forced_labour", "pred_discrimination",
    "pred_freedom_of_association", "pred_working_hours",
    "pred_compensation", "pred_osh", "pred_contracts"
]

# ----------------------------------------------------------------
# 1. Link every article to a factory
#
# Each article may be linked to a factory via:
#   - NER match (osh_factory_matched)
#   - Title match (bhrrc_factory_matched)
# We use whichever is available.
# ----------------------------------------------------------------

news["factory_ref"] = news["osh_factory_matched"].fillna(news["bhrrc_factory_matched"])

# Keep only articles linked to a factory
linked = news[news["factory_ref"].notna()].copy()
print("Articles linked to a factory:", len(linked))
print("Unique factories covered:", linked["factory_ref"].nunique())

# ----------------------------------------------------------------
# 2. Aggregate predictions to factory level
#
# For each factory, across all its linked articles we compute:
#   - Risk prevalence per dimension: proportion of articles
#     flagged for each risk (0.0 to 1.0)
#   - Article count: how many articles reference this factory
#   - Date of most recent article: recency signal
#   - Workers affected: from OSH registry
# ----------------------------------------------------------------

def aggregate_factory(group):
    result = {}
    result["article_count"] = len(group)
    result["latest_article"] = group["date"].max()

    # Risk prevalence per dimension
    for col in pred_cols:
        dim = col.replace("pred_", "")
        result[f"prevalence_{dim}"] = group[col].mean()

    return pd.Series(result)

factory_df = linked.groupby("factory_ref")[list(linked.columns)].apply(
    aggregate_factory, include_groups=False
).reset_index()
factory_df = factory_df.rename(columns={"factory_ref": "factory_name"})

print("\nFactory-level aggregation done:", len(factory_df), "factories")

# ----------------------------------------------------------------
# 3. Merge OSH metadata
#    Bring in number of workers, location, os_id from OSH registry
# ----------------------------------------------------------------

osh_meta = osh[["os_id", "name", "address", "lat", "lng",
                 "number_of_workers", "is_closed"]].copy()
osh_meta["name_lower"] = osh_meta["name"].str.lower().str.strip()
factory_df["name_lower"] = factory_df["factory_name"].str.lower().str.strip()

factory_df = factory_df.merge(osh_meta, on="name_lower", how="left")

# ----------------------------------------------------------------
# 4. Apply ILO/Better Work CAT weights
#
# Weights reflect the severity hierarchy in the Better Work CAT
# and ILO core convention prioritisation:
#
#   HIGH (3): Fundamental rights — child labour, forced labour
#             These are zero-tolerance issues in the Better Work
#             CAT (ILO & IFC, 2025) and non-derogable under
#             ILO Declaration on Fundamental Principles (1998)
#
#   MEDIUM (2): Serious but remediable — discrimination, freedom
#               of association, OSH, working hours
#
#   LOW (1): Contractual/administrative — compensation, contracts
#            Important but typically addressable through dialogue
# ----------------------------------------------------------------

WEIGHTS = {
    "child_labour"          : 3,   # C138, C182 — zero tolerance (CAT)
    "forced_labour"         : 3,   # C29, C105  — zero tolerance (CAT)
    "discrimination"        : 2,   # C100, C111
    "freedom_of_association": 2,   # C87, C98
    "osh"                   : 2,   # C155
    "working_hours"         : 2,   # C1
    "compensation"          : 1,   # C95, C131
    "contracts"             : 1,   # C158
}

TOTAL_WEIGHT = sum(WEIGHTS.values())  # = 16

# ----------------------------------------------------------------
# 5. Compute weighted composite risk score
#
# Formula:
#   composite_score = Σ (prevalence_i × weight_i) / Σ weight_i
#
# Result is a value between 0.0 (no risk) and 1.0 (maximum risk)
# ----------------------------------------------------------------

def compute_composite(row):
    score = 0
    for dim, weight in WEIGHTS.items():
        prevalence = row.get(f"prevalence_{dim}", 0)
        score += prevalence * weight
    return round(score / TOTAL_WEIGHT, 4)

factory_df["raw_score"] = factory_df.apply(compute_composite, axis=1)

# Confidence adjustment: dampen scores from factories with few articles
# using a logarithmic scaling factor. A factory with 1 article gets
# ~50% confidence, 3 articles ~73%, 10+ articles ~92%.
# This prevents single-article factories from dominating the ranking.
factory_df["confidence"] = factory_df["article_count"].apply(
    lambda n: round(1 - 1 / (1 + np.log1p(n)), 4)
)
factory_df["composite_score"] = (
    factory_df["raw_score"] * factory_df["confidence"]
).round(4)

# ----------------------------------------------------------------
# 6. Assign risk bands
#
# Thresholds are defined relative to the weighted score range
# and aligned with Better Work's non-compliance reporting levels:
#   High   : composite >= 0.60
#   Medium : composite >= 0.35
#   Low    : composite < 0.35
# ----------------------------------------------------------------

def assign_band(score):
    if score >= 0.50:
        return "High"
    elif score >= 0.30:
        return "Medium"
    else:
        return "Low"

factory_df["risk_band"] = factory_df["composite_score"].apply(assign_band)

# ----------------------------------------------------------------
# 7. Rank factories from highest to lowest risk
# ----------------------------------------------------------------

# Deduplicate factories with near-identical names by keeping highest score
factory_df["name_lower"] = factory_df["factory_name"].str.lower().str.strip()
factory_df = factory_df.sort_values("composite_score", ascending=False)
factory_df = factory_df.drop_duplicates(subset="name_lower", keep="first").reset_index(drop=True)
factory_df["rank"] = factory_df.index + 1

# ----------------------------------------------------------------
# 8. Summary
# ----------------------------------------------------------------

print("\nRisk band distribution:")
print(factory_df["risk_band"].value_counts())

print("\nTop 10 highest risk factories:")
display_cols = ["rank", "factory_name", "composite_score", "risk_band", "article_count"]
print(factory_df[display_cols].head(10).to_string(index=False))

print("\nComposite score summary:")
print(factory_df["composite_score"].describe().round(3))

# ----------------------------------------------------------------
# 9. Save outputs for dashboard
# ----------------------------------------------------------------

factory_df.to_csv("stage5_factory_risk_index.csv", index=False, encoding="utf-8-sig")

print("\nStage 5 done. Saved to stage5_factory_risk_index.csv")
print("Ready for dashboard development.")

Articles linked to a factory: 313
Unique factories covered: 143

Factory-level aggregation done: 143 factories

Risk band distribution:
risk_band
Low       72
Medium    63
High       8
Name: count, dtype: int64

Top 10 highest risk factories:
 rank                          factory_name  composite_score risk_band  article_count
    1                          Lita Myanmar           0.6969      High             10
    2                         Bohua Fashion           0.6418      High              5
    3 Myanmar Guotai Huasheng Glory Fashion           0.5446      High              3
    4                      Myanmar LNY Caps           0.5397      High              4
    5             Tianjin Fashion Milestone           0.5396      High             12
    6                     Wonderful Apparel           0.5367      High              6
    7                       Sunrise Myanmar           0.5367      High              6
    8                        J-Land Myanmar           0.5204      Hig

In [4]:
import pandas as pd
from thefuzz import process

df = pd.read_csv("stage5_factory_risk_index.csv")

# Fuzzy deduplicate — for each factory, check if a higher-scoring
# factory with a similar name (>=90) already exists, and drop it
names = df["factory_name"].tolist()
to_drop = set()

for i, name in enumerate(names):
    if i in to_drop:
        continue
    for j, other in enumerate(names):
        if i >= j or j in to_drop:
            continue
        score = process.extractOne(name, [other])
        if score and score[1] >= 90:
            # Keep the one with higher composite score (already sorted)
            to_drop.add(j)

df = df[~df.index.isin(to_drop)].reset_index(drop=True)
df["rank"] = df.index + 1

print("Factories after deduplication:", len(df))
print(df[["rank", "factory_name", "composite_score", "risk_band", "article_count"]].head(10).to_string(index=False))

df.to_csv("stage5_factory_risk_index.csv", index=False, encoding="utf-8-sig")
print("Saved.")

Factories after deduplication: 133
 rank                          factory_name  composite_score risk_band  article_count
    1                          Lita Myanmar           0.6969      High             10
    2                         Bohua Fashion           0.6418      High              5
    3 Myanmar Guotai Huasheng Glory Fashion           0.5446      High              3
    4                      Myanmar LNY Caps           0.5397      High              4
    5             Tianjin Fashion Milestone           0.5396      High             12
    6                     Wonderful Apparel           0.5367      High              6
    7                       Sunrise Myanmar           0.5367      High              6
    8                        J-Land Myanmar           0.5204      High              3
    9                       Eternal Fashion           0.4885    Medium              6
   10                           E&P Fashion           0.4720    Medium              3
Saved.
